# History Aware RAG

- In history aware RAG the system looks at the conversation history and rewrites vague or context-dependent questions into clear standalone questions

In [1]:
import os
from dotenv import load_dotenv

print("Exists:", os.path.exists("../openai.env"))

load_dotenv("../openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [2]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [3]:
persistent_directory="db/chroma_db"
embeddings= OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(persist_directory=persistent_directory, embedding_function=embeddings)

In [4]:
model = ChatOpenAI(model = "gpt-4o")

## Store our conversation as messages

In [5]:
chat_history = []

In [12]:
def ask_question(user_question):

    if chat_history:
        messages = [
            SystemMessage(content=(
                "Given the chat history, rewrite the new question to be standalone. "
                "Return only the rewritten question, nothing else."
            ))
        ] + chat_history + [
            HumanMessage(content=f"New Question: {user_question}")
        ]
        search_question = model.invoke(messages).content.strip()
    else:
        search_question = user_question

    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)

    context = "\n".join(f"- {doc.page_content}" for doc in docs)
    combined_input = f"""Based on the following documents, please answer this question: {user_question}
    Documents:
    {context}
    Provide a clear, helpful answer using only the information from these documents.
    If the documents do not contain the answer, say so instead of guessing.
    """

    messages = [
        SystemMessage(content="You are a helpful assistant"),
        HumanMessage(content=combined_input),
    ]
    answer = model.invoke(messages).content

    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    print(answer)

    return answer
        

In [15]:
print(chat_history)

[HumanMessage(content='What is photosynthesis', additional_kwargs={}, response_metadata={}), AIMessage(content='Photosynthesis is a series of biological processes in which photopigment-bearing autotrophic organisms, like most plants, algae, and cyanobacteria, convert light energy—usually from sunlight—into chemical energy necessary to fuel their metabolism. This typically refers to oxygenic photosynthesis, which involves the conversion of light energy into chemical energy with oxygen released as a byproduct through water splitting. Photosynthetic organisms store this energy within the bonds of organic compounds, such as carbohydrates like glucose. When the organism needs energy, it metabolizes these compounds through cellular respiration.\n\nSome organisms perform anoxygenic photosynthesis, which does not produce oxygen. Instead, these organisms might use reductants like hydrogen sulfide, releasing sulfur instead of oxygen.\n\nThe process always begins when light energy is absorbed by 

In [7]:
def start_chat():
    print ("Ask me questions! Type 'quit' to exit.")
    while True:
        question = input("\nYour question: ")

        if question.lower=='quit':
            print("Goodbye")
            break
        ask_question(question)


        

        

In [13]:
def start_chat():
    print("Ask me questions! Type 'quit' to exit.")
    
    while True:
        question = input("\nYour question: ")
        
        if question.lower() == 'quit':
            print("Goodbye!")
            break
            
        ask_question(question)



In [14]:
start_chat()

Ask me questions! Type 'quit' to exit.



Your question:  What is photosynthesis


Photosynthesis is a series of biological processes by which autotrophic organisms, such as most plants, algae, and cyanobacteria, convert light energy—usually from sunlight—into chemical energy. This energy is used to fuel their metabolism. Photosynthesis typically involves the conversion of carbon dioxide and water into sugars and other organic compounds, releasing oxygen as a byproduct. This process primarily takes place in chloroplasts in plants and algae and involves pigments like chlorophyll, which absorb light.

There are two main types of photosynthesis: oxygenic and anoxygenic. Oxygenic photosynthesis is the most common form and releases oxygen. It involves light-dependent reactions, where light energy is used to produce ATP and reduced nicotinamide adenine dinucleotide phosphate (NADPH), and light-independent reactions (the Calvin cycle), where carbon dioxide is converted into sugars. Anoxygenic photosynthesis does not release oxygen and is performed by certain bacteria that u


Your question:  what are the ingredients needed for photosynthesis


The ingredients needed for oxygenic photosynthesis, as outlined in the documents, are carbon dioxide (CO2), water (H2O), and light energy (photons). These components are utilized in the photosynthetic process to produce carbohydrates (such as glucose) and oxygen (O2) as a byproduct.



Your question:  quit


Goodbye!
